In [1]:
# 1. Upload ZIP file
from google.colab import files
uploaded = files.upload()

# 2. Get uploaded filename
filename = list(uploaded.keys())[0]

# 3. Extract ZIP file
import zipfile

with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall("dataset")

# 4. Find and load CSV
import glob
import pandas as pd

csv_file = glob.glob("dataset/*.csv")[0]
data = pd.read_csv(csv_file)

print(data.head())
print("Dataset size:", data.shape)

Saving WELFake_Dataset.csv.zip to WELFake_Dataset.csv.zip
   Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3           3  Bobby Jindal, raised Hindu, uses story of Chri...   
4           4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the demonstrators gathered last ...      1  
3  A dozen politically active pastors came here f...      0  
4  The RS-28 Sarmat missile, dubbed Satan 2, will...      1  
Dataset size: (72134, 4)


In [2]:
# 5. Prepare text
data["title"] = data["title"].fillna("")
data["text"] = data["text"].fillna("")
data["content"] = data["title"] + " " + data["text"]

In [3]:
# 6. Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data["content"],
    data["label"],
    test_size=0.2,
    random_state=42
)

In [4]:
# 7. TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

In [5]:
# 8. Train Logistic Regression
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [6]:
# 9. Check accuracy
from sklearn.metrics import accuracy_score

prediction = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, prediction))

Accuracy: 0.9479448256740833


In [7]:
from sklearn.metrics import f1_score
print("F1-Score:", f1_score(y_test, prediction))

F1-Score: 0.9490950993018369


In [8]:
# 10. Predict new news
news = input("Enter news: ")

news = tfidf.transform([news])
result = model.predict(news)[0]

if result == 0:
    print("FAKE NEWS")
else:
    print("REAL NEWS")

Enter news: PAPER
REAL NEWS


In [9]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
accuracy = accuracy_score(y_test, prediction)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)

Accuracy: 0.9479448256740833
Precision: 0.9441672285906946
Recall: 0.9540746797492505
F1-Score: 0.9490950993018369


In [10]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 83.2 MB/s eta 0:00:00


In [11]:
%%writefile app.py

import streamlit as st
import numpy as np

st.set_page_config(
    page_title="Fake News Detection",
    page_icon="📰",
    layout="wide"
)

st.title("📰 Fake News Detection Dashboard")
st.write("Fake News Detection using TF-IDF and Logistic Regression")

# Model Performance
st.subheader("Model Performance")

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("Accuracy", f"{accuracy:.2%}")

with col2:
    st.metric("Precision", f"{precision:.2%}")

with col3:
    st.metric("Recall", f"{recall:.2%}")

with col4:
    st.metric("F1 Score", f"{f1:.2%}")

st.divider()

# News Input
st.subheader("Enter News Article")

news = st.text_area(
    "Enter the news text below:",
    height=200
)

if st.button("🔍 Detect News"):

    if news.strip() == "":
        st.warning("Please enter a news article.")

    else:
        # Convert news into TF-IDF
        news_tfidf = tfidf.transform([news])

        # Prediction
        result = model.predict(news_tfidf)[0]

        # Confidence
        probability = model.predict_proba(news_tfidf)[0]
        confidence = max(probability) * 100

        st.divider()

        if result == 0:
            st.error("🚨 FAKE NEWS")
        else:
            st.success("✅ REAL NEWS")

        st.metric(
            "Prediction Confidence",
            f"{confidence:.2f}%"
        )

Writing app.py


In [12]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Model and TF-IDF saved successfully!")

Model and TF-IDF saved successfully!


In [13]:
%%writefile app.py

import streamlit as st
import pickle

# Load trained model and TF-IDF
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

with open("tfidf.pkl", "rb") as f:
    tfidf = pickle.load(f)

st.set_page_config(
    page_title="Fake News Detection",
    page_icon="📰",
    layout="wide"
)

st.title("📰 Fake News Detection Dashboard")
st.write("Fake News Detection using TF-IDF and Logistic Regression")

st.divider()

# News input
st.subheader("Enter News Article")

news = st.text_area(
    "Enter the news text below:",
    height=200
)

if st.button("🔍 Detect News"):

    if news.strip() == "":
        st.warning("Please enter a news article.")

    else:
        # Convert news into TF-IDF
        news_tfidf = tfidf.transform([news])

        # Make prediction
        result = model.predict(news_tfidf)[0]

        # Get confidence
        probability = model.predict_proba(news_tfidf)[0]
        confidence = max(probability) * 100

        st.divider()

        if result == 0:
            st.error("🚨 FAKE NEWS")
        else:
            st.success("✅ REAL NEWS")

        st.metric(
            "Prediction Confidence",
            f"{confidence:.2f}%"
        )

Overwriting app.py


In [14]:
!streamlit run app.py &>/content/logs.txt &

!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) ^C


In [15]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [16]:
!cat /content/logs.txt



2026-10-13 05:33:58.820 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.70.82.51:8501

  Stopping...


In [17]:
!streamlit run app.py --server.port 8501 > /content/logs.txt 2>&1 &

In [18]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [19]:
!tail -30 /content/logs.txt



2026-10-13 05:34:12.594 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.70.82.51:8501



In [20]:
!pkill -f "streamlit run" || true

^C


In [21]:
!streamlit run app.py \
--server.port 8501 \
--server.address 0.0.0.0 \
--server.enableCORS false \
--server.enableXsrfProtection false \
> /content/logs.txt 2>&1 &

In [22]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [23]:
import pickle

# Save evaluation results
metrics_data = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "y_test": y_test,
    "prediction": prediction
}

with open("metrics.pkl", "wb") as f:
    pickle.dump(metrics_data, f)

print("Evaluation metrics saved successfully!")

Evaluation metrics saved successfully!


In [24]:
%%writefile app.py

import streamlit as st
import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# ============================================================
# LOAD MODEL, TF-IDF AND EVALUATION RESULTS
# ============================================================

with open("model.pkl", "rb") as f:
    model = pickle.load(f)

with open("tfidf.pkl", "rb") as f:
    tfidf = pickle.load(f)

with open("metrics.pkl", "rb") as f:
    metrics_data = pickle.load(f)

accuracy = metrics_data["accuracy"]
precision = metrics_data["precision"]
recall = metrics_data["recall"]
f1 = metrics_data["f1"]

y_test = metrics_data["y_test"]
prediction = metrics_data["prediction"]


# ============================================================
# PAGE SETTINGS
# ============================================================

st.set_page_config(
    page_title="Fake News Detection Dashboard",
    page_icon="📰",
    layout="wide"
)


# ============================================================
# TITLE
# ============================================================

st.title("📰 Fake News Detection Dashboard")

st.write(
    "Fake News Detection using TF-IDF and Logistic Regression"
)

st.divider()


# ============================================================
# MODEL PERFORMANCE
# ============================================================

st.subheader("📊 Model Performance")

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        "Accuracy",
        f"{accuracy * 100:.2f}%"
    )

with col2:
    st.metric(
        "Precision",
        f"{precision * 100:.2f}%"
    )

with col3:
    st.metric(
        "Recall",
        f"{recall * 100:.2f}%"
    )

with col4:
    st.metric(
        "F1 Score",
        f"{f1 * 100:.2f}%"
    )


st.divider()


# ============================================================
# CONFUSION MATRIX
# ============================================================

st.subheader("📈 Confusion Matrix")

cm = confusion_matrix(y_test, prediction)

fig, ax = plt.subplots()

ax.imshow(cm)

ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

ax.set_xticklabels(["Fake", "Real"])
ax.set_yticklabels(["Fake", "Real"])

# Display values inside the matrix
for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

st.pyplot(fig)

st.write(
    "The confusion matrix shows the number of correctly "
    "and incorrectly classified Fake and Real news articles."
)


st.divider()


# ============================================================
# NEWS PREDICTION
# ============================================================

st.subheader("📰 Enter News Article")

news = st.text_area(
    "Enter the news text below:",
    height=200,
    placeholder="Enter a news article here..."
)


# ============================================================
# DETECT NEWS BUTTON
# ============================================================

if st.button("🔍 Detect News"):

    if news.strip() == "":
        st.warning("Please enter a news article.")

    else:

        # Convert news into TF-IDF
        news_tfidf = tfidf.transform([news])

        # Make prediction
        result = model.predict(news_tfidf)[0]

        # Get prediction probabilities
        probability = model.predict_proba(news_tfidf)[0]

        # Highest probability
        confidence = max(probability) * 100


        st.divider()

        st.subheader("🔎 Prediction Result")


        # ====================================================
        # FAKE / REAL RESULT
        # ====================================================

        if result == 0:

            st.error("🚨 FAKE NEWS")

        else:

            st.success("✅ REAL NEWS")


        # ====================================================
        # CONFIDENCE
        # ====================================================

        st.metric(
            "Prediction Confidence",
            f"{confidence:.2f}%"
        )

        # Progress bar
        st.progress(
            int(confidence)
        )


        # ====================================================
        # EXPLANATION
        # ====================================================

        st.info(
            "The prediction is generated using the trained "
            "TF-IDF and Logistic Regression model."
        )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "Machine Learning Project | Fake News Detection using NLP, "
    "TF-IDF and Logistic Regression"
)

Overwriting app.py


In [25]:
!pkill -f "streamlit run" || true

^C


In [26]:
!streamlit run app.py \
--server.port 8501 \
--server.address 0.0.0.0 \
--server.enableCORS false \
--server.enableXsrfProtection false \
> /content/logs.txt 2>&1 &

In [27]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [28]:
import streamlit as st
import pandas as pd
import os
import glob
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


# =========================================================
# PAGE CONFIGURATION
# =========================================================

st.set_page_config(
    page_title="Fake News Detection",
    page_icon="📰",
    layout="wide"
)


# =========================================================
# TITLE
# =========================================================

st.title("📰 Fake News Detection Dashboard")

st.write(
    "Fake News Detection using TF-IDF and Logistic Regression"
)

st.divider()


# =========================================================
# MODEL FILES
# =========================================================

MODEL_FILE = "model.pkl"
VECTORIZER_FILE = "tfidf.pkl"


# =========================================================
# LOAD OR TRAIN MODEL
# =========================================================

@st.cache_resource
def load_model():

    # -----------------------------------------------------
    # If saved model already exists, load it
    # -----------------------------------------------------

    if os.path.exists(MODEL_FILE) and os.path.exists(VECTORIZER_FILE):

        with open(MODEL_FILE, "rb") as f:
            model = pickle.load(f)

        with open(VECTORIZER_FILE, "rb") as f:
            vectorizer = pickle.load(f)

        return model, vectorizer


    # -----------------------------------------------------
    # Find CSV dataset
    # -----------------------------------------------------

    csv_files = []

    possible_locations = [
        "*.csv",
        "dataset/*.csv",
        "/content/*.csv",
        "/content/dataset/*.csv"
    ]

    for location in possible_locations:
        csv_files.extend(glob.glob(location))


    if len(csv_files) == 0:

        st.error(
            "Dataset CSV file was not found. "
            "Please upload your WELFake dataset CSV file."
        )

        st.stop()


    # -----------------------------------------------------
    # Load dataset
    # -----------------------------------------------------

    dataset_file = csv_files[0]

    data = pd.read_csv(dataset_file)


    # -----------------------------------------------------
    # Check required columns
    # -----------------------------------------------------

    if "title" not in data.columns or "text" not in data.columns:
        st.error(
            "Dataset must contain 'title' and 'text' columns."
        )
        st.stop()


    if "label" not in data.columns:
        st.error(
            "Dataset must contain a 'label' column."
        )
        st.stop()


    # -----------------------------------------------------
    # Handle missing values
    # -----------------------------------------------------

    data["title"] = data["title"].fillna("")
    data["text"] = data["text"].fillna("")


    # -----------------------------------------------------
    # Combine title and text
    # -----------------------------------------------------

    data["content"] = data["title"] + " " + data["text"]


    # -----------------------------------------------------
    # Remove empty rows
    # -----------------------------------------------------

    data = data[data["content"].str.strip() != ""]


    # -----------------------------------------------------
    # Features and target
    # -----------------------------------------------------

    X = data["content"]
    y = data["label"]


    # -----------------------------------------------------
    # Train-test split
    # -----------------------------------------------------

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )


    # -----------------------------------------------------
    # TF-IDF
    # -----------------------------------------------------

    vectorizer = TfidfVectorizer(
        max_features=5000
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)


    # -----------------------------------------------------
    # Logistic Regression
    # -----------------------------------------------------

    model = LogisticRegression(
        max_iter=1000
    )

    model.fit(
        X_train_tfidf,
        y_train
    )


    # -----------------------------------------------------
    # Save model and vectorizer
    # -----------------------------------------------------

    with open(MODEL_FILE, "wb") as f:
        pickle.dump(model, f)

    with open(VECTORIZER_FILE, "wb") as f:
        pickle.dump(vectorizer, f)


    return model, vectorizer


# =========================================================
# LOAD MODEL
# =========================================================

model, vectorizer = load_model()


# =========================================================
# USER INPUT
# =========================================================

st.header("Enter News Article")

st.write("Enter the news text below:")

news = st.text_area(
    "News Article",
    height=200,
    placeholder="Paste or type a news article here..."
)


# =========================================================
# DETECT NEWS BUTTON
# =========================================================

if st.button(
    "🔍 Detect News",
    use_container_width=False
):

    # -----------------------------------------------------
    # Check empty input
    # -----------------------------------------------------

    if news.strip() == "":

        st.warning(
            "Please enter a news article."
        )

    else:

        # -------------------------------------------------
        # Convert input into TF-IDF
        # -------------------------------------------------

        news_tfidf = vectorizer.transform([news])


        # -------------------------------------------------
        # Prediction
        # -------------------------------------------------

        prediction = model.predict(
            news_tfidf
        )[0]


        # -------------------------------------------------
        # Prediction probability
        # -------------------------------------------------

        probability = model.predict_proba(
            news_tfidf
        )[0]

        confidence = max(probability) * 100


        # -------------------------------------------------
        # Display result
        # -------------------------------------------------

        if prediction == 0:

            st.error(
                "❌ FAKE NEWS"
            )

            st.subheader("Prediction Confidence")

            st.write(
                f"{confidence:.2f}%"
            )

        else:

            st.success(
                "✅ REAL NEWS"
            )

            st.subheader("Prediction Confidence")

            st.write(
                f"{confidence:.2f}%"
            )


# =========================================================
# FOOTER
# =========================================================

st.divider()

st.caption(
    "Machine Learning Project | TF-IDF + Logistic Regression"
)

2026-10-13 05:34:43.675 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-13 05:34:43.677 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-13 05:34:43.840 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-13 05:34:43.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-13 05:34:43.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-13 05:34:43.845 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-13 05:34:43.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator()

In [29]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/logs.txt 2>&1 &

In [30]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>